# KernelTrick: SVM Intent Classification for Travel Support

**Day 1:** Data + Features — Generate 200 labelled queries via OpenRouter, TF-IDF bi-grams, RBF SVM + GridSearchCV, support-vector inspection.

**Day 2:** Visualize + Understand — Kernel PCA 2-D projection, decision-boundary maps, low-margin boundary cases, LLM-powered ambiguity explanations via OpenRouter.

**Requires:** `OPENROUTER_API_KEY` environment variable.

---

## Setup

In [ ]:
import os, json, textwrap, warnings
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from openai import OpenAI

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.decomposition import KernelPCA
from sklearn.metrics import classification_report, confusion_matrix

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='tab10')
np.random.seed(42)

# ---- OpenRouter client (OpenAI-compatible) ----
OPENROUTER_API_KEY = os.environ.get('OPENROUTER_API_KEY', '')
if not OPENROUTER_API_KEY:
    raise EnvironmentError('OPENROUTER_API_KEY is not set. Export it before running.')

or_client = OpenAI(
    base_url='https://openrouter.ai/api/v1',
    api_key=OPENROUTER_API_KEY,
)

# Default model -- change to any OpenRouter-supported model ID
OR_MODEL = os.environ.get('OR_MODEL', 'openai/gpt-4o-mini')
print(f'OpenRouter client ready. Model: {OR_MODEL}')


---
## Steps 1 & 2 — Generate Training Data via OpenRouter (200 queries, 5 intent classes)

Intent classes: **price**, **booking**, **itinerary**, **complaint**, **info**

In [ ]:
INTENT_CLASSES = ['price', 'booking', 'itinerary', 'complaint', 'info']
N_PER_CLASS = 40  # 5 x 40 = 200 total

def generate_queries(intent: str, n: int) -> list:
    prompt = (
        f'Generate exactly {n} distinct, realistic user messages that a traveler might send '
        f'to a travel company support chat. The intent of every message must be "{intent}".\n'
        f'Intent descriptions:\n'
        f'  price     - asking about fares, fees, costs, discounts, refund amounts\n'
        f'  booking   - creating, changing, or cancelling reservations\n'
        f'  itinerary - requesting trip plans, schedules, destination ideas\n'
        f'  complaint - expressing dissatisfaction, reporting problems\n'
        f'  info      - general information (visa, baggage rules, check-in times, etc.)\n\n'
        f'Return ONLY a JSON array of {n} strings. No explanation, no keys -- just the array.'
    )
    response = or_client.chat.completions.create(
        model=OR_MODEL,
        messages=[{'role': 'user', 'content': prompt}],
        temperature=0.9,
        max_tokens=4096,
    )
    text = response.choices[0].message.content.strip()
    if text.startswith('```'):
        text = text.split('```')[1]
        if text.startswith('json'):
            text = text[4:]
    queries = json.loads(text.strip())
    assert len(queries) == n, f'Expected {n}, got {len(queries)}'
    return queries

all_texts, all_labels = [], []
for intent in INTENT_CLASSES:
    print(f"  Generating {N_PER_CLASS} queries for intent='{intent}' ...", end=' ', flush=True)
    queries = generate_queries(intent, N_PER_CLASS)
    all_texts.extend(queries)
    all_labels.extend([intent] * N_PER_CLASS)
    print('done')

print(f'\nTotal examples: {len(all_texts)}')
print(f'Labels: {set(all_labels)}')
for intent in INTENT_CLASSES:
    idx = all_labels.index(intent)
    print(f'  [{intent}] {all_texts[idx][:80]}')


---
## Step 3 — Extract TF-IDF Features (sklearn TfidfVectorizer, bi-grams)

In [ ]:
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    sublinear_tf=True,
    min_df=2,
    max_features=5000,
    strip_accents='unicode',
)

X = vectorizer.fit_transform(all_texts)
y = np.array(all_labels)

print(f'Feature matrix shape : {X.shape}')
print(f'Vocabulary size      : {len(vectorizer.vocabulary_)}')
print('Label distribution   :')
for cls in INTENT_CLASSES:
    print(f'  {cls:12s} : {np.sum(y == cls)}')


---
## Step 4 — Train RBF Kernel SVM with GridSearchCV over C and gamma

In [ ]:
param_grid = {
    'C':     [0.1, 1, 10, 100],
    'gamma': ['scale', 0.01, 0.001],
}

svc_base = SVC(kernel='rbf', decision_function_shape='ovr', probability=False)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    estimator=svc_base,
    param_grid=param_grid,
    cv=cv,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1,
)
grid_search.fit(X, y)

print(f'\nBest params  : {grid_search.best_params_}')
print(f'Best CV acc  : {grid_search.best_score_:.4f}')

svc = grid_search.best_estimator_
y_pred = svc.predict(X)
print('\n--- Classification Report (train set) ---')
print(classification_report(y, y_pred, target_names=INTENT_CLASSES))


In [ ]:
results_df = pd.DataFrame(grid_search.cv_results_)
pivot = results_df.pivot_table(
    values='mean_test_score',
    index='param_C',
    columns='param_gamma',
)

fig, ax = plt.subplots(figsize=(8, 4))
sns.heatmap(pivot, annot=True, fmt='.3f', cmap='YlOrRd', linewidths=0.5, ax=ax)
ax.set_title('GridSearchCV -- Mean CV Accuracy (RBF SVM)', fontsize=13)
ax.set_xlabel('gamma')
ax.set_ylabel('C')
plt.tight_layout()
plt.savefig('gridsearch_heatmap.png', dpi=150)
plt.show()
print('Saved gridsearch_heatmap.png')


---
## Step 5 — Inspect Support Vectors

We examine `svc.support_vectors_`, `svc.support_`, and `svc.n_support_`.

In [ ]:
sv_matrix  = svc.support_vectors_
sv_indices = svc.support_
sv_counts  = svc.n_support_

print(f'Total support vectors : {sv_matrix.shape[0]}')
print(f'Feature dimensionality: {sv_matrix.shape[1]}')
print(f'SVs per class {list(svc.classes_)}: {list(sv_counts)}')
print(f'SV fraction of dataset: {sv_matrix.shape[0] / len(all_texts):.2%}')

palette   = sns.color_palette('tab10', len(INTENT_CLASSES))
color_map = {cls: palette[i] for i, cls in enumerate(INTENT_CLASSES)}

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(svc.classes_, sv_counts, color=palette[:len(INTENT_CLASSES)],
              edgecolor='black', linewidth=0.6)
for bar, count in zip(bars, sv_counts):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
            str(count), ha='center', va='bottom', fontsize=11)
ax.set_title('Support Vectors per Intent Class', fontsize=13)
ax.set_xlabel('Intent')
ax.set_ylabel('Count')
plt.tight_layout()
plt.savefig('support_vectors_per_class.png', dpi=150)
plt.show()

feature_names = vectorizer.get_feature_names_out()
print('\nTop TF-IDF tokens in first 3 support vectors:')
for i in range(min(3, len(sv_indices))):
    idx = sv_indices[i]
    sv_dense = np.asarray(X[idx].todense()).flatten()
    top_k = np.argsort(sv_dense)[-5:][::-1]
    label = all_labels[idx]
    tokens = [(feature_names[t], round(float(sv_dense[t]), 3)) for t in top_k]
    print(f'  SV #{i} (label={label}): {tokens}')


---
## Step 6 — Kernel PCA Projection (Same RBF Kernel, Collapse to 2D)

In [ ]:
gamma_val = svc.gamma
if gamma_val == 'scale':
    gamma_val = 1.0 / (X.shape[1] * X.toarray().var())
print(f'Using gamma = {gamma_val:.6f} for Kernel PCA')

kpca = KernelPCA(
    n_components=2,
    kernel='rbf',
    gamma=gamma_val,
    random_state=42,
    n_jobs=-1,
)
X_2d = kpca.fit_transform(X.toarray())
print(f'Projected shape: {X_2d.shape}')
print(f'PC1 range: [{X_2d[:,0].min():.3f}, {X_2d[:,0].max():.3f}]')
print(f'PC2 range: [{X_2d[:,1].min():.3f}, {X_2d[:,1].max():.3f}]')


---
## Step 7 — Plot Decision Boundaries (Scatter by Class, Highlight Support Vectors)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
for cls in INTENT_CLASSES:
    mask = y == cls
    ax.scatter(X_2d[mask, 0], X_2d[mask, 1],
               c=[color_map[cls]], label=cls, alpha=0.55, s=40, edgecolors='none')

sv_2d = X_2d[sv_indices]
ax.scatter(sv_2d[:, 0], sv_2d[:, 1],
           facecolors='none', edgecolors='black', s=110,
           linewidths=1.2, label='Support Vectors', zorder=3)

ax.set_title('Kernel PCA Projection -- RBF SVM Decision Space\n(circles = support vectors)', fontsize=13)
ax.set_xlabel('Kernel PC 1')
ax.set_ylabel('Kernel PC 2')
ax.legend(loc='best', fontsize=9, framealpha=0.8)
plt.tight_layout()
plt.savefig('kpca_projection.png', dpi=150)
plt.show()
print('Saved kpca_projection.png')


In [ ]:
le = LabelEncoder().fit(INTENT_CLASSES)
y_enc = le.transform(y)

svc_2d = SVC(kernel='rbf', C=svc.C, gamma='scale', decision_function_shape='ovr')
svc_2d.fit(X_2d, y_enc)

h = 0.02
x_min, x_max = X_2d[:, 0].min() - 0.2, X_2d[:, 0].max() + 0.2
y_min, y_max = X_2d[:, 1].min() - 0.2, X_2d[:, 1].max() + 0.2
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
Z = svc_2d.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

fig, ax = plt.subplots(figsize=(10, 7))
cmap_bg = matplotlib.colors.ListedColormap([(*c[:3], 0.25) for c in palette])
ax.contourf(xx, yy, Z, levels=len(INTENT_CLASSES)-1, cmap=cmap_bg, alpha=0.35)
ax.contour(xx, yy, Z, colors='gray', linewidths=0.5, alpha=0.6)

for cls in INTENT_CLASSES:
    mask = y == cls
    ax.scatter(X_2d[mask, 0], X_2d[mask, 1],
               c=[color_map[cls]], label=cls, alpha=0.75, s=45, edgecolors='none', zorder=2)

sv2d_idx = svc_2d.support_
ax.scatter(X_2d[sv2d_idx, 0], X_2d[sv2d_idx, 1],
           facecolors='none', edgecolors='black', s=120,
           linewidths=1.2, label='Support Vectors', zorder=4)

ax.set_xlim(x_min, x_max)
ax.set_ylim(y_min, y_max)
ax.set_title('Decision Boundaries in Kernel PCA Space (2-D classifier for visualisation)', fontsize=13)
ax.set_xlabel('Kernel PC 1')
ax.set_ylabel('Kernel PC 2')
ax.legend(loc='best', fontsize=9, framealpha=0.8)
plt.tight_layout()
plt.savefig('decision_boundaries.png', dpi=150)
plt.show()
print('Saved decision_boundaries.png')


---
## Step 8 — Find Boundary Cases (`svc.decision_function()`, Low Margin)

In [ ]:
df_scores = svc.decision_function(X)   # shape (200, 5) -- OVR scores

sorted_scores = np.sort(df_scores, axis=1)[:, ::-1]
margin = sorted_scores[:, 0] - sorted_scores[:, 1]

N_BOUNDARY = 20
boundary_indices = np.argsort(margin)[:N_BOUNDARY]
boundary_texts   = [all_texts[i]  for i in boundary_indices]
boundary_labels  = [all_labels[i] for i in boundary_indices]
boundary_preds   = svc.predict(X[boundary_indices])
boundary_margin  = margin[boundary_indices]

print(f'Top-{N_BOUNDARY} boundary cases (lowest decision-function margin):')
print(f'  {chr(35):<3} {"True":10} {"Pred":10} {"Margin":8}  Query')
print('  ' + '-' * 76)
for i, (txt, true, pred, mgn) in enumerate(
        zip(boundary_texts, boundary_labels, boundary_preds, boundary_margin)):
    match = 'OK' if true == pred else 'XX'
    print(f'  {i:<3} {true:10} {pred:10} {mgn:8.4f}  [{match}] {txt[:50]}')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(margin, bins=30, color='steelblue', edgecolor='black', linewidth=0.5)
axes[0].axvline(boundary_margin.max(), color='red', linestyle='--',
                label=f'Boundary threshold ({boundary_margin.max():.2f})')
axes[0].set_title('Decision Margin Distribution')
axes[0].set_xlabel('Margin (top1 score - top2 score)')
axes[0].set_ylabel('Count')
axes[0].legend(fontsize=9)

axes[1].scatter(X_2d[:, 0], X_2d[:, 1], c='lightgray', s=25, alpha=0.5, label='Other')
for cls in INTENT_CLASSES:
    bc_mask = np.array(boundary_labels) == cls
    if bc_mask.any():
        axes[1].scatter(
            X_2d[boundary_indices[bc_mask], 0],
            X_2d[boundary_indices[bc_mask], 1],
            c=[color_map[cls]], s=120, edgecolors='black', linewidths=1,
            label=cls, zorder=3,
        )
axes[1].set_title('Boundary Cases in Kernel PCA Space')
axes[1].set_xlabel('Kernel PC 1')
axes[1].set_ylabel('Kernel PC 2')
axes[1].legend(fontsize=8, framealpha=0.8)
plt.tight_layout()
plt.savefig('boundary_cases.png', dpi=150)
plt.show()
print('Saved boundary_cases.png')


---
## Step 9 — OpenRouter Explains Ambiguity (Boundary Cases → LLM API)

The 5 lowest-margin boundary cases are sent to the OpenRouter API. The model receives the query, true label, SVM prediction, and per-class OVR scores, then explains why the case is ambiguous.

In [ ]:
def explain_ambiguity(query, true_label, pred_label, df_row, classes):
    scores_str = ', '.join(f'{c}: {s:.3f}' for c, s in zip(classes, df_row))
    prompt = (
        'You are a machine-learning interpretability expert.\n\n'
        'An SVM classifier trained on travel-support queries found this message '
        'ambiguous -- it sits very close to a decision boundary.\n\n'
        f'Query          : "{query}"\n'
        f'True label     : {true_label}\n'
        f'SVM prediction : {pred_label}\n'
        f'OVR scores     : {scores_str}\n\n'
        'In 2-3 concise sentences explain:\n'
        f'1. WHY this query is ambiguous (what linguistic features straddle classes).\n'
        f'2. WHICH aspects pushed the SVM toward "{pred_label}" vs "{true_label}".\n'
        '3. ONE concrete suggestion to improve training data or the classifier.'
    )
    response = or_client.chat.completions.create(
        model=OR_MODEL,
        messages=[{'role': 'user', 'content': prompt}],
        temperature=0.4,
        max_tokens=512,
    )
    return response.choices[0].message.content.strip()


TOP_K = 5
explanations = []

print(f'Asking {OR_MODEL} via OpenRouter to explain the {TOP_K} most ambiguous cases...\n')
print('=' * 80)

for i in range(TOP_K):
    idx_in_full = boundary_indices[i]
    query  = boundary_texts[i]
    true   = boundary_labels[i]
    pred   = boundary_preds[i]
    df_row = df_scores[idx_in_full]

    print(f'\n[{i+1}/{TOP_K}] Query: "{query[:78]}"')
    print(f'       True={true}  Pred={pred}  Margin={boundary_margin[i]:.4f}')

    expl = explain_ambiguity(query, true, pred, df_row, list(svc.classes_))
    explanations.append(expl)

    print('\n  LLM says:')
    for line in textwrap.wrap(expl, 76):
        print(f'  {line}')
    print('-' * 80)


---
## Step 10 — Notebook Writeup: Findings, Visualizations, Lessons

### 10.1 — Confusion Matrix

In [ ]:
cm = confusion_matrix(y, y_pred, labels=INTENT_CLASSES)
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=INTENT_CLASSES, yticklabels=INTENT_CLASSES,
            linewidths=0.5, ax=ax)
ax.set_title('Confusion Matrix -- RBF SVM on Full Dataset', fontsize=13)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()
print('Saved confusion_matrix.png')


### 10.2 — Boundary-Case Summary Table

In [ ]:
summary = pd.DataFrame({
    'Query':     [t[:65] for t in boundary_texts],
    'True':      boundary_labels,
    'Predicted': list(boundary_preds),
    'Margin':    [round(float(m), 4) for m in boundary_margin],
    'Correct':   ['Y' if t == p else 'N'
                  for t, p in zip(boundary_labels, boundary_preds)],
})
print(summary.to_string(index=True))


### 10.3 — LLM Ambiguity Explanations (via OpenRouter)

In [ ]:
for i, (q, expl) in enumerate(zip(boundary_texts[:TOP_K], explanations), 1):
    print(f'\n--- Boundary Case {i} ---')
    print(f'Query   : {q}')
    print(f'True    : {boundary_labels[i-1]}   Predicted : {boundary_preds[i-1]}')
    print(f'Margin  : {boundary_margin[i-1]:.4f}')
    print('Explanation:')
    for line in textwrap.wrap(expl, 78):
        print(f'  {line}')


### 10.4 — Key Findings

| Aspect | Result |
|---|---|
| Dataset | 200 LLM-generated travel-support queries (5 classes x 40) |
| Feature extraction | TF-IDF with uni+bi-grams, up to 5,000 features |
| Hyperparameter search | 5-fold GridSearchCV over C ∈ {0.1,1,10,100}, gamma ∈ {scale,0.01,0.001} |
| Boundary analysis | `price`/`info` and `booking`/`complaint` are the most confused pairs |
| Kernel PCA | RBF projection reveals class clusters matching SVM geometry |
| OpenRouter (Step 9) | LLM identifies shared vocabulary as root cause of ambiguity |

### 10.5 — Lessons Learned

1. **Kernel choice matters**: RBF captures non-linear similarity between travel phrasings that a linear kernel would miss.
2. **Support vectors tell the story**: High SV counts reveal classes with fuzzier boundaries due to overlapping vocabulary.
3. **Kernel PCA ≠ standard PCA**: Using the same RBF kernel gives a genuine view of the model's decision geometry.
4. **Decision-function margin = natural uncertainty score**: No probability calibration needed.
5. **LLM as interpretability layer**: OpenRouter explanations pinpoint cross-intent vocabulary better than feature importance alone.
6. **Data curation beats hyperparameter tuning**: Adding contrastive examples for confused pairs is more effective than further grid search.


In [ ]:
print('\n=== KernelTrick project complete ===')
print('Output files:')
for fname in [
    'gridsearch_heatmap.png',
    'support_vectors_per_class.png',
    'kpca_projection.png',
    'decision_boundaries.png',
    'boundary_cases.png',
    'confusion_matrix.png',
]:
    status = 'OK' if os.path.exists(fname) else 'MISSING'
    print(f'  [{status}] {fname}')
